# LeetCode 576 — Out of Boundary Paths bằng Python

Link đề: https://leetcode.com/problems/out-of-boundary-paths/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming (quy hoạch động), đếm đường đi trên lưới

Skeleton LeetCode:

```python
class Solution:
    def findPaths(self, m: int, n: int, maxMove: int, startRow: int, startColumn: int) -> int:
        ...
```

Ghi chú: đây là bài đếm số đường đi, đáp án rất lớn nên phải lấy dư `10^9 + 7`. Cái khó nhất là nhận ra **số đường đi tăng theo cấp số nhân (`4^k`)** nhưng chỉ có `m·n·k` trạng thái khác nhau — nên có thể gom lại bằng quy hoạch động. Notebook đi từ đệ quy thuần, đệ quy có nhớ (memoization), bảng DP lùi, DP tiến cuốn chiếu, tới nhân ma trận cho `maxMove` cực lớn.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: quả bóng đứng ở ô `(startRow, startColumn)`, mỗi lần đi sang 1 trong 4 ô kề (có thể đi ra ngoài lưới), tối đa `maxMove` lần; đếm số **đường đi** đưa bóng ra khỏi lưới, lấy dư `10^9 + 7`.
2. Nắm các công cụ Python cần dùng: phép lấy dư `%` và vì sao phải lấy dư, mảng hướng đi `dirs`, kiểm tra biên `0 <= i < m`, đệ quy, `functools.cache`/`lru_cache`, tạo mảng 2D đúng cách (tránh `[[0]*n]*m`).
3. Đi từ **đệ quy thuần (`4^k`)** → **đệ quy có nhớ** → **bảng DP lùi** → **DP tiến cuốn chiếu `O(mn)` bộ nhớ** → **nhân ma trận** cho `maxMove` rất lớn, và hiểu vì sao các cách cho cùng đáp án.
4. Hiểu mấu chốt: số đường đi từ một ô với `k` bước còn lại **chỉ phụ thuộc vào `(ô, k)`**, không phụ thuộc quá khứ; và một đường đi **kết thúc ngay khi ra khỏi lưới** (không đi tiếp được nữa), nên mỗi đường chỉ được đếm một lần.
5. Tránh các lỗi kinh điển: kiểm tra `k == 0` trước khi kiểm tra ra ngoài biên, memo thiếu `k`, quên `%`, khởi tạo mảng 2D bị dùng chung, hiểu nhầm "tối đa `maxMove`" thành "đúng `maxMove`".

## 2. Đề bài & ví dụ

Có một lưới `m x n` và một quả bóng đặt ở ô `[startRow, startColumn]`. Bạn được di chuyển quả bóng sang một trong bốn ô kề (trên, dưới, trái, phải); bóng **có thể đi ra ngoài lưới** (vượt qua biên). Bạn được di chuyển **tối đa** `maxMove` lần.

Cho năm số nguyên `m, n, maxMove, startRow, startColumn`, hãy trả về **số đường đi** đưa quả bóng ra khỏi biên lưới. Vì đáp án có thể rất lớn, trả về **kết quả modulo `10^9 + 7`**.

**Ví dụ 1**

- Input: `m = 2, n = 2, maxMove = 2, startRow = 0, startColumn = 0`
- Output: `6`

**Ví dụ 2**

- Input: `m = 1, n = 3, maxMove = 3, startRow = 0, startColumn = 1`
- Output: `12`

**Ràng buộc:**

- `1 <= m, n <= 50`
- `0 <= maxMove <= 50`
- `0 <= startRow < m`
- `0 <= startColumn < n`

Làm rõ "đường đi": một đường đi là **dãy các hướng** (lên/xuống/trái/phải). Hai dãy hướng khác nhau là hai đường khác nhau, kể cả khi chúng đi qua cùng tập ô. Đường đi **dừng ngay khi bóng ra ngoài** — bóng đã ra khỏi lưới thì không quay lại. Số bước dùng ít hơn `maxMove` vẫn hợp lệ ("tối đa").

In [1]:
from typing import List
from functools import lru_cache
from itertools import product
import random

MOD = 10**9 + 7

examples = [
    # (m, n, maxMove, startRow, startColumn, expected)
    (2, 2, 2, 0, 0, 6),
    (1, 3, 3, 0, 1, 12),
]
for m, n, k, r, c, expected in examples:
    print(f"lưới {m}x{n}, tối đa {k} bước, bắt đầu ({r},{c}) -> kỳ vọng {expected}")

lưới 2x2, tối đa 2 bước, bắt đầu (0,0) -> kỳ vọng 6
lưới 1x3, tối đa 3 bước, bắt đầu (0,1) -> kỳ vọng 12


## 3. Kiến thức nền

### 3.1. Phép lấy dư `%` và vì sao phải `mod`

`a % b` là phần dư khi chia `a` cho `b`: `17 % 5 == 2`. Khi đáp án quá lớn, đề yêu cầu lấy dư cho `10^9 + 7` (một số nguyên tố lớn quen thuộc) để kết quả nằm trong 32-bit của các ngôn ngữ như Java/C++.

Tính chất quan trọng: **lấy dư từng bước rồi cộng nhân** cho kết quả giống lấy dư cuối cùng:

```
(a + b) % M == ((a % M) + (b % M)) % M
```

Nhờ vậy ta lấy dư ngay trong lúc cộng dồn, số luôn nhỏ. (Python có số nguyên lớn tuỳ ý nên không tràn, nhưng số quá lớn sẽ chậm; ngoài ra thói quen lấy dư từng bước cần cho Java/C++.)

In [2]:
MOD = 10**9 + 7
print(17 % 5)                             # 2
print((MOD + 1) % MOD)                    # 1
a, b = 10**9 + 5, 10**9 + 6
print((a + b) % MOD, ((a % MOD) + (b % MOD)) % MOD)   # hai cách giống nhau

2
1
1000000004 1000000004


### 3.2. Bốn hướng đi: mảng `dirs`

Thay vì viết tay 4 trường hợp, ta lưu độ lệch `(dr, dc)` của 4 hướng:

```python
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # lên, xuống, trái, phải
for dr, dc in DIRS:
    x, y = i + dr, j + dc
```

Lời giải của một số tài liệu dùng mẹo `dirs = (-1, 0, 1, 0, -1)` rồi lấy từng cặp liền kề `(dirs[d], dirs[d+1])` (`pairwise`): các cặp `(-1,0), (0,1), (1,0), (0,-1)` chính là bốn hướng. Hai cách tương đương; ta dùng danh sách cặp cho dễ đọc.

In [3]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
i, j = 1, 1
print([(i + dr, j + dc) for dr, dc in DIRS])    # 4 ô kề của (1,1)

# mẹo dirs = (-1,0,1,0,-1)
dirs = (-1, 0, 1, 0, -1)
print([(dirs[d], dirs[d + 1]) for d in range(4)])

[(0, 1), (2, 1), (1, 0), (1, 2)]
[(-1, 0), (0, 1), (1, 0), (0, -1)]


### 3.3. Kiểm tra biên: `0 <= i < m`

Python cho viết **so sánh dây chuyền**: `0 <= i < m` nghĩa là `0 <= i and i < m`. Ô nằm trong lưới khi cả hàng và cột trong khoảng hợp lệ; **ra ngoài** khi vi phạm một trong hai.

Ví dụ lưới `2 x 2`: `(0, 1)` ở trong; `(-1, 0)`, `(2, 0)`, `(0, 2)`, `(0, -1)` ở ngoài.

In [4]:
m, n = 2, 2

def inside(i, j):
    return 0 <= i < m and 0 <= j < n

for cell in [(0, 1), (-1, 0), (2, 0), (0, 2), (0, -1)]:
    print(cell, "trong" if inside(*cell) else "NGOÀI")

(0, 1) trong
(-1, 0) NGOÀI
(2, 0) NGOÀI
(0, 2) NGOÀI
(0, -1) NGOÀI


### 3.4. Đệ quy và đệ quy có nhớ (`functools.cache`)

Một hàm đệ quy gọi lại chính nó với bài toán nhỏ hơn. Khi cùng một lời gọi `f(x)` bị lặp lại nhiều lần, ta có thể **nhớ** kết quả (memoization): lần đầu tính và lưu, những lần sau lấy ra ngay.

Python hỗ trợ sẵn bằng decorator `@lru_cache(None)` (hoặc `@cache` từ Python 3.9). Ví dụ Fibonacci: không nhớ thì tính lại rất nhiều lần; có nhớ thì mỗi `n` chỉ tính một lần.

In [5]:
calls = 0
def fib_plain(n):
    global calls
    calls += 1
    return n if n < 2 else fib_plain(n - 1) + fib_plain(n - 2)

calls_memo = 0
@lru_cache(None)
def fib_memo(n):
    global calls_memo
    calls_memo += 1
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)

print(fib_plain(20), "- số lần gọi:", calls)            # hàng chục nghìn lần
print(fib_memo(20), "- số lần gọi:", calls_memo)        # chỉ 21 lần

6765 - số lần gọi: 21891
6765 - số lần gọi: 21


### 3.5. Tạo mảng 2D/3D đúng cách (tránh dùng chung hàng)

`[[0] * n] * m` **không** tạo `m` hàng độc lập: nó lặp lại *cùng một* danh sách `m` lần, nên sửa một hàng là sửa tất cả. Cách đúng là list comprehension:

```python
grid = [[0] * n for _ in range(m)]
```

Mỗi vòng của comprehension tạo một danh sách mới.

In [6]:
m, n = 3, 2

bad = [[0] * n] * m
bad[0][0] = 7
print("sai :", bad)          # cả 3 hàng đều đổi!

good = [[0] * n for _ in range(m)]
good[0][0] = 7
print("đúng:", good)         # chỉ hàng 0 đổi

sai : [[7, 0], [7, 0], [7, 0]]
đúng: [[7, 0], [0, 0], [0, 0]]


## 4. Ý tưởng nền tảng / trực giác

Hãy tính tay Ví dụ 2: lưới `1 x 3` (một hàng, ba cột), bóng ở giữa `(0,1)`, tối đa 3 bước. Vì chỉ có 1 hàng, đi **lên** hoặc **xuống** là ra ngoài ngay; ô ở rìa trái/phải có thêm một lối ra theo chiều ngang.

**Bước 1.** Từ `(0,1)`: lên/xuống → ra ngoài (**2 đường**, mỗi đường dài 1 bước). Sang trái → `(0,0)`; sang phải → `(0,2)` (mỗi bên 1 cách đi).

**Bước 2.** Từ `(0,0)`: lên, xuống, trái → ra ngoài (3 lối), phải → `(0,1)`. Tương tự từ `(0,2)`: 3 lối ra, 1 bước về giữa. Có 1 cách đang ở `(0,0)` và 1 cách ở `(0,2)`, nên có `1·3 + 1·3 = 6` đường ra ở bước này. Còn ở trong: 2 cách đang ở `(0,1)`.

**Bước 3.** Từ `(0,1)` còn 2 cách đang ở đó, mỗi cách có 2 lối ra → `2·2 = 4` đường.

**Tổng:** `2 + 6 + 4 = 12`. Khớp đáp án.

Nhận xét: ta không cần liệt kê từng dãy hướng. Chỉ cần biết **"có bao nhiêu cách đang đứng ở mỗi ô sau mỗi bước"**. Đó là mầm mống của quy hoạch động.

In [7]:
# Tính tay Ví dụ 2 bằng code mini (m=1, n=3): ways[j] = số cách đang đứng ở cột j sau t bước
ways = [0, 1, 0]            # bắt đầu ở cột 1
total_exits = 0
for step in range(1, 4):
    new = [0, 0, 0]
    exits = 0
    for j in range(3):
        if ways[j] == 0:
            continue
        exits += 2 * ways[j]                  # lên / xuống: ra ngoài ngay
        if j == 0: exits += ways[j]           # trái: ra ngoài
        else: new[j - 1] += ways[j]
        if j == 2: exits += ways[j]           # phải: ra ngoài
        else: new[j + 1] += ways[j]
    total_exits += exits
    print(f"sau bước {step}: ways = {new}, số đường ra ở bước này = {exits}")
    ways = new
print("tổng =", total_exits)   # 12

sau bước 1: ways = [1, 0, 1], số đường ra ở bước này = 2
sau bước 2: ways = [0, 2, 0], số đường ra ở bước này = 6
sau bước 3: ways = [2, 0, 2], số đường ra ở bước này = 4
tổng = 12


## 5. Cách 1 — Brute Force: đệ quy thuần

### Ý tưởng

Định nghĩa `dfs(i, j, k)` = số đường đi ra khỏi lưới nếu bóng đang ở `(i, j)` và **còn `k` bước**. Cách tính:

1. Nếu `(i, j)` **đã ở ngoài** lưới: đã thành công, trả `1`. (Chỉ được gọi khi còn `k >= 0` bước vì mỗi lời gọi con tiêu 1 bước từ `k >= 1`.)
2. Ngược lại, nếu `k == 0` (còn trong lưới và hết bước): trả `0`.
3. Ngược lại: cộng `dfs` của 4 ô kề với `k - 1`.

Thứ tự (1) rồi (2) rất quan trọng (xem mục 13): bước đi cuối cùng chạm ra ngoài vẫn được tính.

### Code

In [8]:
def find_paths_brute(m, n, maxMove, startRow, startColumn):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def dfs(i, j, k):
        if not (0 <= i < m and 0 <= j < n):
            return 1                 # đã ra ngoài: một đường hợp lệ
        if k == 0:
            return 0                 # còn trong lưới nhưng hết bước
        total = 0
        for dr, dc in DIRS:
            total += dfs(i + dr, j + dc, k - 1)
        return total % MOD

    return dfs(startRow, startColumn, maxMove)

for m, n, k, r, c, expected in examples:
    got = find_paths_brute(m, n, k, r, c)
    print(got, expected, "OK" if got == expected else "SAI")

6 6 OK
12 12 OK


### Độ phức tạp và nút thắt

- **Time:** `O(4^k)` — mỗi bước rẽ 4 nhánh, sâu `k` tầng. Với `k = 50` là `4^50 ≈ 10^30`: không bao giờ chạy xong.
- **Space:** `O(k)` cho ngăn xếp đệ quy.

**Nút thắt:** cùng một bộ `(i, j, k)` bị tính lại vô số lần (đi "lên rồi xuống" và "xuống rồi lên" đều dẫn về cùng một ô với cùng số bước còn lại). Ta cần **nhớ** kết quả. Hãy đếm số lần gọi để thấy rõ: với lưới `3x3`, `k = 8`, số lời gọi của cách này lên tới hơn mười hai nghìn, trong khi số bộ `(i, j, k)` khác nhau thực sự chạm tới chỉ khoảng ba chục (tối đa `3·3·8 = 72`) — xem cell dưới.

In [9]:
def count_calls_brute(m, n, maxMove, sr, sc):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    calls = 0
    def dfs(i, j, k):
        nonlocal calls
        calls += 1
        if not (0 <= i < m and 0 <= j < n):
            return 1
        if k == 0:
            return 0
        return sum(dfs(i + dr, j + dc, k - 1) for dr, dc in DIRS)
    ans = dfs(sr, sc, maxMove)
    return ans, calls

ans, calls = count_calls_brute(3, 3, 8, 1, 1)
print("đáp án =", ans, "| số lời gọi =", calls, "| số trạng thái khác nhau tối đa = 3*3*9 =", 3 * 3 * 9)

đáp án = 3508 | số lời gọi = 12869 | số trạng thái khác nhau tối đa = 3*3*9 = 81


## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái đủ để dự đoán tương lai.** Số đường ra ngoài từ lúc này chỉ phụ thuộc vào hai thứ: **bóng đang ở ô nào** và **còn bao nhiêu bước**. Cách bóng đi đến đây (lịch sử) không quan trọng. Do đó `dfs(i, j, k)` chỉ có `m · n · (maxMove + 1)` giá trị khác nhau — nhiều nhất `50 · 50 · 51 ≈ 127.500` — rất nhỏ so với `4^50`.

Hai cách tận dụng:

- **Đệ quy có nhớ (top-down):** giữ nguyên hàm `dfs`, thêm bộ nhớ đệm. Mỗi trạng thái tính đúng một lần, mỗi lần tốn `O(4)`.
- **Bảng DP (bottom-up):** liệt kê trạng thái theo thứ tự `k = 0, 1, 2, ...`; `f[k]` chỉ cần `f[k-1]`.

Còn một góc nhìn **đối ngẫu** nữa — *DP tiến*: thay vì hỏi "từ ô này có bao nhiêu đường ra", hỏi "sau `t` bước có bao nhiêu **cách đang đứng** ở mỗi ô", và mỗi khi một bước đưa bóng ra ngoài thì cộng cách đó vào đáp án. Cách này chỉ cần hai lớp `m x n` (bộ nhớ `O(mn)`).

Đoạn code dưới minh hoạ ý "chỉ phụ thuộc `(ô, k)`": đếm số trạng thái khác nhau mà đệ quy thật sự chạm tới.

In [10]:
def distinct_states(m, n, maxMove, sr, sc):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    seen = set()
    def dfs(i, j, k):
        if not (0 <= i < m and 0 <= j < n) or k == 0:
            return
        if (i, j, k) in seen:
            return
        seen.add((i, j, k))
        for dr, dc in DIRS:
            dfs(i + dr, j + dc, k - 1)
    dfs(sr, sc, maxMove)
    return len(seen)

print(distinct_states(3, 3, 8, 1, 1), "<= 3*3*8 =", 3 * 3 * 8)
print(distinct_states(50, 50, 50, 25, 25), "<= 50*50*50 =", 50 * 50 * 50)

32 <= 3*3*8 = 72
31875 <= 50*50*50 = 125000


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ (top-down)

**Giải thích từng đoạn.**

1. Giữ nguyên `dfs(i, j, k)` của cách 1.
2. Bọc bằng `@lru_cache(None)` (hoặc dùng `dict`/mảng 3 chiều tự quản lý). **Khoá bộ nhớ gồm cả `k`**, không chỉ `(i, j)`.
3. Mỗi trạng thái chỉ tính một lần, độ phức tạp `O(m·n·k)`.

**Code demo:**

In [11]:
def find_paths_memo(m, n, maxMove, startRow, startColumn):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    @lru_cache(None)
    def dfs(i, j, k):
        if not (0 <= i < m and 0 <= j < n):
            return 1
        if k == 0:
            return 0
        total = 0
        for dr, dc in DIRS:
            total = (total + dfs(i + dr, j + dc, k - 1)) % MOD
        return total

    return dfs(startRow, startColumn, maxMove)

for m, n, k, r, c, expected in examples:
    print(find_paths_memo(m, n, k, r, c), expected)
print("50x50, 50 bước:", find_paths_memo(50, 50, 50, 25, 25))

6 6
12 12
50x50, 50 bước: 276775132


**Code nộp LeetCode (Cách 2):**

In [12]:
class SolutionMemo:
    def findPaths(self, m: int, n: int, maxMove: int, startRow: int, startColumn: int) -> int:
        mod = 10**9 + 7
        dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]

        @lru_cache(None)
        def dfs(i: int, j: int, k: int) -> int:
            if not (0 <= i < m and 0 <= j < n):
                return 1
            if k == 0:
                return 0
            ans = 0
            for dr, dc in dirs:
                ans = (ans + dfs(i + dr, j + dc, k - 1)) % mod
            return ans

        return dfs(startRow, startColumn, maxMove)

print(SolutionMemo().findPaths(2, 2, 2, 0, 0))   # 6

6


**Độ phức tạp (Cách 2):** Time `O(m·n·k)` (mỗi trạng thái tốn `O(4)`), Space `O(m·n·k)` cho bộ nhớ đệm + `O(k)` ngăn xếp đệ quy (độ sâu tối đa `k + 1 ≤ 51`, không lo tràn).

Giải thích vì sao đúng ngắn gọn: `dfs(i,j,k)` = tổng `dfs` của 4 hàng xóm với `k-1`, trong đó hàng xóm nằm ngoài lưới đóng góp đúng 1 (đường dừng tại đó). Đây là phép đếm theo **bước đi đầu tiên** — phân chia các đường đi thành 4 nhóm rời nhau, đầy đủ.

### 7.2. Cách 3 — Bảng DP lùi (bottom-up) `f[k][i][j]`

**Giải thích từng đoạn.**

1. `f[k][i][j]` = số đường ra ngoài từ `(i, j)` khi còn `k` bước — chính là `dfs(i, j, k)`.
2. `f[0][i][j] = 0` với mọi ô trong lưới (hết bước mà chưa ra).
3. Với `k = 1..maxMove`: `f[k][i][j] = Σ` trên 4 hướng của `1` (nếu ô kề ở ngoài) hoặc `f[k-1][x][y]` (nếu ô kề ở trong).
4. Đáp án: `f[maxMove][startRow][startColumn]`.

Không còn đệ quy: ta điền bảng theo thứ tự `k` tăng dần, nên khi tính `f[k]` thì `f[k-1]` đã có đủ.

**Code demo:**

In [13]:
def find_paths_table(m, n, maxMove, startRow, startColumn):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    # f[k][i][j]: số đường ra từ (i, j) khi còn k bước
    f = [[[0] * n for _ in range(m)] for _ in range(maxMove + 1)]
    for k in range(1, maxMove + 1):
        for i in range(m):
            for j in range(n):
                total = 0
                for dr, dc in DIRS:
                    x, y = i + dr, j + dc
                    if 0 <= x < m and 0 <= y < n:
                        total += f[k - 1][x][y]
                    else:
                        total += 1                # bước này đưa bóng ra ngoài
                f[k][i][j] = total % MOD
    return f[maxMove][startRow][startColumn]

for m, n, k, r, c, expected in examples:
    print(find_paths_table(m, n, k, r, c), expected)

6 6
12 12


**Code nộp LeetCode (Cách 3):**

In [14]:
class SolutionTable:
    def findPaths(self, m: int, n: int, maxMove: int, startRow: int, startColumn: int) -> int:
        mod = 10**9 + 7
        dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        f = [[[0] * n for _ in range(m)] for _ in range(maxMove + 1)]
        for k in range(1, maxMove + 1):
            for i in range(m):
                for j in range(n):
                    total = 0
                    for dr, dc in dirs:
                        x, y = i + dr, j + dc
                        if 0 <= x < m and 0 <= y < n:
                            total += f[k - 1][x][y]
                        else:
                            total += 1
                    f[k][i][j] = total % mod
        return f[maxMove][startRow][startColumn]

print(SolutionTable().findPaths(1, 3, 3, 0, 1))   # 12

12


**Độ phức tạp (Cách 3):** Time `O(m·n·k)`, Space `O(m·n·k)` (bảng 3 chiều). Nhận xét: `f[k]` chỉ dùng `f[k-1]` nên thực ra chỉ cần hai lớp — nhưng bảng này tính *cho mọi ô xuất phát* cùng lúc (hữu ích nếu có nhiều truy vấn khác nhau về ô bắt đầu).

### 7.3. Cách 4 — DP tiến cuốn chiếu (lời giải nộp): đếm "cách đang đứng ở mỗi ô"

**Giải thích từng đoạn.**

1. `dp[i][j]` = số cách đi từ ô bắt đầu để **vẫn đang ở trong lưới** và đứng ở `(i, j)` sau đúng `t` bước. Ban đầu `t = 0`: `dp[startRow][startColumn] = 1`.
2. Mỗi bước `t → t + 1`: tạo bảng mới `nxt` toàn 0. Với mỗi ô có `dp[i][j] > 0` và mỗi hướng:
   - nếu ô kề **ngoài** lưới: có `dp[i][j]` đường ra ở bước này → cộng vào `ans`;
   - nếu ô kề **trong** lưới: `nxt[x][y] += dp[i][j]`.
3. Lặp `maxMove` bước; `ans` cộng dồn mọi lần ra ngoài ở bất kỳ bước nào (đúng nghĩa "tối đa `maxMove` bước").

Điểm hay: bóng đã ra ngoài **không còn nằm trong `dp`** nên không bị đếm lại. Mỗi đường đi được tính đúng một lần, tại thời điểm nó ra ngoài. Chỉ cần hai bảng `m x n`.

**Code demo:**

In [15]:
def find_paths_forward(m, n, maxMove, startRow, startColumn):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    dp = [[0] * n for _ in range(m)]
    dp[startRow][startColumn] = 1
    ans = 0
    for _ in range(maxMove):
        nxt = [[0] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                ways = dp[i][j]
                if ways == 0:
                    continue
                for dr, dc in DIRS:
                    x, y = i + dr, j + dc
                    if 0 <= x < m and 0 <= y < n:
                        nxt[x][y] = (nxt[x][y] + ways) % MOD
                    else:
                        ans = (ans + ways) % MOD
        dp = nxt
    return ans

for m, n, k, r, c, expected in examples:
    print(find_paths_forward(m, n, k, r, c), expected)

6 6
12 12


**Code nộp LeetCode (Cách 4):**

In [16]:
class SolutionForward:
    def findPaths(self, m: int, n: int, maxMove: int, startRow: int, startColumn: int) -> int:
        mod = 10**9 + 7
        dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        dp = [[0] * n for _ in range(m)]
        dp[startRow][startColumn] = 1
        ans = 0
        for _ in range(maxMove):
            nxt = [[0] * n for _ in range(m)]
            for i in range(m):
                for j in range(n):
                    ways = dp[i][j]
                    if ways == 0:
                        continue
                    for dr, dc in dirs:
                        x, y = i + dr, j + dc
                        if 0 <= x < m and 0 <= y < n:
                            nxt[x][y] = (nxt[x][y] + ways) % mod
                        else:
                            ans = (ans + ways) % mod
            dp = nxt
        return ans

print(SolutionForward().findPaths(2, 2, 2, 0, 0))   # 6

6


**Độ phức tạp (Cách 4):** Time `O(m·n·k)`, Space `O(m·n)` (hai lớp). Đây là cách tiết kiệm bộ nhớ nhất trong các cách `O(mnk)`.

### 7.4. Cách 5 — Nhân ma trận (cho `maxMove` cực lớn)

**Giải thích từng đoạn.** Mỗi bước `t → t+1` là một phép biến đổi **tuyến tính** trên vector `dp` (độ dài `m·n`): `dp' = A · dp` với `A` là ma trận kề (ô `u` → ô `v` kề nhau thì `A[v][u] = 1`). Ta thêm một trạng thái hấp thụ `E` ("tổng số đường đã ra") nhận `ways` từ mỗi lối ra: `A[E][u] = số hướng từ u đi ra ngoài`, và `A[E][E] = 1` (giữ nguyên tổng). Khi đó vector `(dp, ans)` sau `k` bước là `A^k` nhân vector ban đầu, và `A^k` tính bằng **luỹ thừa nhanh** trong `O(log k)` phép nhân ma trận.

Với đề này (`k ≤ 50`) cách này chậm hơn DP thường (ma trận `2501 x 2501` không hề nhỏ). Nhưng nó cho thấy nếu `maxMove` lên tới `10^18` mà lưới nhỏ, DP tuyến tính không còn dùng được và nhân ma trận là lối thoát. Độ phức tạp `O((mn)^3 · log k)`.

**Code demo (và code nộp — chạy đúng với lưới nhỏ, đủ để đối chiếu):**

In [17]:
def mat_mul(A, B):
    n, p, q = len(A), len(B), len(B[0])
    C = [[0] * q for _ in range(n)]
    for i in range(n):
        Ai = A[i]
        Ci = C[i]
        for t in range(p):
            a = Ai[t]
            if a:
                Bt = B[t]
                for j in range(q):
                    Ci[j] = (Ci[j] + a * Bt[j]) % MOD
    return C

def mat_pow(A, e):
    size = len(A)
    R = [[int(i == j) for j in range(size)] for i in range(size)]   # ma trận đơn vị
    while e:
        if e & 1:
            R = mat_mul(R, A)
        A = mat_mul(A, A)
        e >>= 1
    return R

def find_paths_matrix(m, n, maxMove, startRow, startColumn):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    size = m * n + 1                       # m*n ô + 1 trạng thái 'đã ra'
    E = m * n
    A = [[0] * size for _ in range(size)]
    for i in range(m):
        for j in range(n):
            u = i * n + j
            for dr, dc in DIRS:
                x, y = i + dr, j + dc
                if 0 <= x < m and 0 <= y < n:
                    A[x * n + y][u] += 1   # u -> v
                else:
                    A[E][u] += 1           # u -> ra ngoài
    A[E][E] = 1                            # trạng thái hấp thụ giữ nguyên
    P = mat_pow(A, maxMove)
    s = startRow * n + startColumn
    return P[E][s] % MOD

for m, n, k, r, c, expected in examples:
    print(find_paths_matrix(m, n, k, r, c), expected)
# maxMove khổng lồ trên lưới nhỏ (DP thường không chạy nổi):
print(find_paths_matrix(2, 2, 10**18, 0, 0))

6 6
12 12
438952511


In [18]:
class SolutionMatrix:
    def findPaths(self, m: int, n: int, maxMove: int, startRow: int, startColumn: int) -> int:
        mod = 10**9 + 7
        dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        size = m * n + 1
        E = m * n
        A = [[0] * size for _ in range(size)]
        for i in range(m):
            for j in range(n):
                u = i * n + j
                for dr, dc in dirs:
                    x, y = i + dr, j + dc
                    if 0 <= x < m and 0 <= y < n:
                        A[x * n + y][u] += 1
                    else:
                        A[E][u] += 1
        A[E][E] = 1

        def mul(X, Y):
            R = [[0] * size for _ in range(size)]
            for a in range(size):
                for t in range(size):
                    v = X[a][t]
                    if v:
                        Yt = Y[t]
                        Ra = R[a]
                        for b in range(size):
                            Ra[b] = (Ra[b] + v * Yt[b]) % mod
            return R

        R = [[int(a == b) for b in range(size)] for a in range(size)]
        e = maxMove
        while e:
            if e & 1:
                R = mul(R, A)
            A = mul(A, A)
            e >>= 1
        return R[E][startRow * n + startColumn] % mod

print(SolutionMatrix().findPaths(1, 3, 3, 0, 1))   # 12 (chỉ dùng cho lưới nhỏ)

12


**Độ phức tạp (Cách 5):** Time `O((mn)^3 · log k)`, Space `O((mn)^2)`. Chỉ có ý nghĩa khi `k` rất lớn và `mn` nhỏ; với ràng buộc của đề, hãy dùng Cách 4.

## 8. TRACE từng bước

Trace Ví dụ 1 bằng DP tiến: lưới `2 x 2`, `maxMove = 2`, bóng ở `(0,0)`. Mỗi ô có 2 hướng ra ngoài (góc), 2 hướng vào ô kề trong lưới.

| bước t | `dp` (số cách đang đứng) trước bước | số đường ra ở bước này | `ans` sau bước |
|---|---|---|---|
| 1 | `[[1,0],[0,0]]` | từ `(0,0)`: lên, trái → 2 | 2 |
| 2 | `[[0,1],[1,0]]` | từ `(0,1)`: lên, phải → 2; từ `(1,0)`: xuống, trái → 2; tổng 4 | 6 |

Tổng `6`. Đúng.

Hàm `trace_forward` in từng bước cho input bất kỳ: bảng `dp` trước bước, số đường ra ở bước, `ans` cộng dồn, và bảng `dp` sau bước.

In [19]:
def trace_forward(m, n, maxMove, startRow, startColumn):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    dp = [[0] * n for _ in range(m)]
    dp[startRow][startColumn] = 1
    ans = 0
    print(f"lưới {m}x{n}, tối đa {maxMove} bước, bắt đầu ({startRow},{startColumn})")
    for t in range(1, maxMove + 1):
        nxt = [[0] * n for _ in range(m)]
        exits = 0
        for i in range(m):
            for j in range(n):
                ways = dp[i][j]
                if ways == 0:
                    continue
                for dr, dc in DIRS:
                    x, y = i + dr, j + dc
                    if 0 <= x < m and 0 <= y < n:
                        nxt[x][y] += ways
                    else:
                        exits += ways
        ans += exits
        print(f"--- bước {t}: dp trước = {dp} | ra ngoài ở bước này = {exits} | ans = {ans}")
        print(f"            dp sau  = {nxt}")
        dp = nxt
    return ans % MOD

print("kết quả:", trace_forward(2, 2, 2, 0, 0))

lưới 2x2, tối đa 2 bước, bắt đầu (0,0)
--- bước 1: dp trước = [[1, 0], [0, 0]] | ra ngoài ở bước này = 2 | ans = 2
            dp sau  = [[0, 1], [1, 0]]
--- bước 2: dp trước = [[0, 1], [1, 0]] | ra ngoài ở bước này = 4 | ans = 6
            dp sau  = [[2, 0], [0, 2]]
kết quả: 6


Trace thứ hai: Ví dụ 2 (`1 x 3`, 3 bước, bóng ở giữa) — khớp phép tính tay ở mục 4 (`2 + 6 + 4 = 12`).

In [20]:
print("kết quả:", trace_forward(1, 3, 3, 0, 1))

lưới 1x3, tối đa 3 bước, bắt đầu (0,1)
--- bước 1: dp trước = [[0, 1, 0]] | ra ngoài ở bước này = 2 | ans = 2
            dp sau  = [[1, 0, 1]]
--- bước 2: dp trước = [[1, 0, 1]] | ra ngoài ở bước này = 6 | ans = 8
            dp sau  = [[0, 2, 0]]
--- bước 3: dp trước = [[0, 2, 0]] | ra ngoài ở bước này = 4 | ans = 12
            dp sau  = [[2, 0, 2]]
kết quả: 12


Ta cũng trace **đệ quy có nhớ** để thấy chỗ cache giúp ích: in mỗi lời gọi `dfs(i, j, k)`, đánh dấu `(cache hit)` khi trạng thái đã tính từ trước.

In [21]:
def trace_memo(m, n, maxMove, startRow, startColumn):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    memo = {}
    stats = {"calls": 0, "hits": 0}

    def dfs(i, j, k, depth):
        stats["calls"] += 1
        pad = "  " * depth
        if not (0 <= i < m and 0 <= j < n):
            print(f"{pad}dfs({i},{j},{k}) -> ngoài lưới = 1")
            return 1
        if k == 0:
            print(f"{pad}dfs({i},{j},{k}) -> hết bước = 0")
            return 0
        if (i, j, k) in memo:
            stats["hits"] += 1
            print(f"{pad}dfs({i},{j},{k}) = {memo[(i, j, k)]} (cache hit)")
            return memo[(i, j, k)]
        total = 0
        for dr, dc in DIRS:
            total += dfs(i + dr, j + dc, k - 1, depth + 1)
        memo[(i, j, k)] = total
        print(f"{pad}dfs({i},{j},{k}) = {total}")
        return total

    res = dfs(startRow, startColumn, maxMove, 0)
    print("kết quả:", res, "| số lời gọi:", stats["calls"], "| cache hit:", stats["hits"])
    return res

trace_memo(2, 2, 2, 0, 0)

  dfs(-1,0,1) -> ngoài lưới = 1
    dfs(0,0,0) -> hết bước = 0
    dfs(2,0,0) -> ngoài lưới = 1
    dfs(1,-1,0) -> ngoài lưới = 1
    dfs(1,1,0) -> hết bước = 0
  dfs(1,0,1) = 2
  dfs(0,-1,1) -> ngoài lưới = 1
    dfs(-1,1,0) -> ngoài lưới = 1
    dfs(1,1,0) -> hết bước = 0
    dfs(0,0,0) -> hết bước = 0
    dfs(0,2,0) -> ngoài lưới = 1
  dfs(0,1,1) = 2
dfs(0,0,2) = 6
kết quả: 6 | số lời gọi: 13 | cache hit: 0


6

## 9. Code nộp LeetCode

Lời giải chuẩn: DP tiến cuốn chiếu (Cách 4). Đúng `class Solution`, đúng signature. (Cách đệ quy có nhớ ở mục 7.1 cũng được chấp nhận, cùng độ phức tạp thời gian.)

In [22]:
from typing import List

class Solution:
    def findPaths(self, m: int, n: int, maxMove: int, startRow: int, startColumn: int) -> int:
        mod = 10**9 + 7
        dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        # dp[i][j]: số cách đang đứng ở (i, j) (vẫn trong lưới) sau t bước
        dp = [[0] * n for _ in range(m)]
        dp[startRow][startColumn] = 1
        ans = 0
        for _ in range(maxMove):
            nxt = [[0] * n for _ in range(m)]
            for i in range(m):
                for j in range(n):
                    ways = dp[i][j]
                    if ways == 0:
                        continue
                    for dr, dc in dirs:
                        x, y = i + dr, j + dc
                        if 0 <= x < m and 0 <= y < n:
                            nxt[x][y] = (nxt[x][y] + ways) % mod
                        else:
                            ans = (ans + ways) % mod      # bước này đưa bóng ra ngoài
            dp = nxt
        return ans

sol = Solution()
for m, n, k, r, c, expected in examples:
    got = sol.findPaths(m, n, k, r, c)
    print(got, expected, "OK" if got == expected else "SAI")

6 6 OK
12 12 OK


## 10. Vì sao đúng

**Định nghĩa.** Một đường đi là dãy hướng `d_1, d_2, ..., d_t` (`1 ≤ t ≤ maxMove`) sao cho bóng ở trong lưới sau `d_1..d_{s}` với mọi `s < t` và **ra ngoài** sau `d_t`. Mỗi đường như vậy xác định duy nhất bởi dãy hướng; ta cần đếm chúng.

**Phân loại theo thời điểm ra ngoài.** Gọi `E_t` là số đường đi ra ngoài *tại đúng bước `t`*. Đáp án `= Σ_{t=1}^{maxMove} E_t`. Hai đường khác thời điểm ra là khác nhau, và một đường chỉ có một thời điểm ra duy nhất, nên tổng không đếm trùng.

**Quy nạp cho `dp`.** Gọi `W_t(i, j)` = số dãy hướng độ dài `t` mà bóng **luôn ở trong lưới** sau mỗi bước và kết thúc ở `(i, j)`. Khi đó:

- `W_0(i, j) = 1` nếu `(i, j)` là ô xuất phát, ngược lại `0`.
- `W_{t+1}(x, y) = Σ_{(i,j) kề (x,y)} W_t(i, j)` (với `(x, y)` trong lưới): mọi dãy độ dài `t+1` ở trong lưới và kết thúc tại `(x,y)` là dãy độ dài `t` kết thúc tại một ô kề rồi thêm một bước.
- `E_{t+1} = Σ_{(i,j)} W_t(i, j) · (số hướng từ (i,j) dẫn ra ngoài)`.

Code trong `Solution` làm đúng ba công thức này. Vì `W_t` **chỉ tính** các dãy còn ở trong lưới nên bóng đã ra ngoài không bị đếm tiếp.

**Vì sao tính modulo từng bước vẫn đúng.** Cả ba công thức chỉ gồm phép cộng (và nhân với hằng nhỏ), và `(a + b) mod M` không phụ thuộc cách nhóm, nên lấy dư sau mỗi phép cộng cho cùng kết quả như lấy dư ở cuối.

Cell dưới kiểm chứng định nghĩa này bằng một **oracle độc lập**: liệt kê *mọi dãy hướng độ dài `maxMove`*, cắt mỗi dãy tại lần ra ngoài đầu tiên rồi đếm số dãy bị cắt **khác nhau** (đó chính là số đường đi theo định nghĩa của đề).

In [23]:
def count_by_enumeration(m, n, maxMove, startRow, startColumn):
    # Oracle độc lập: dùng product để liệt kê mọi dãy hướng độ dài maxMove
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    paths = set()
    for seq in product(range(4), repeat=maxMove):
        i, j = startRow, startColumn
        taken = []
        for d in seq:
            dr, dc = DIRS[d]
            i, j = i + dr, j + dc
            taken.append(d)
            if not (0 <= i < m and 0 <= j < n):
                paths.add(tuple(taken))      # cắt tại lần ra ngoài đầu tiên
                break
    return len(paths)

for m, n, k, r, c, expected in examples:
    print(count_by_enumeration(m, n, k, r, c), expected)

# So với DP trên nhiều lưới nhỏ
for m in range(1, 4):
    for n in range(1, 4):
        for k in range(0, 6):
            for r in range(m):
                for c in range(n):
                    assert count_by_enumeration(m, n, k, r, c) == Solution().findPaths(m, n, k, r, c)
print("Oracle liệt kê khớp Solution trên mọi lưới 1..3 x 1..3, k <= 5")

6 6
12 12


Oracle liệt kê khớp Solution trên mọi lưới 1..3 x 1..3, k <= 5


## 11. Time / Space Complexity

Với `m, n ≤ 50` và `k = maxMove ≤ 50`:

- **Time:** `O(m · n · k)` — `k` bước, mỗi bước duyệt `m·n` ô và 4 hướng. Cụ thể khoảng `50 · 2500 · 4 = 500.000` phép — chạy tức thì.
- **Space:** `O(m · n)` — hai bảng `m x n` (`dp` và `nxt`).

So sánh: đệ quy thuần `O(4^k)`; đệ quy có nhớ và bảng DP lùi `O(m·n·k)` cả time lẫn space; nhân ma trận `O((mn)^3 log k)`.

In [24]:
import time
t0 = time.perf_counter(); a1 = SolutionForward().findPaths(50, 50, 50, 25, 25); t1 = time.perf_counter()
a2 = SolutionMemo().findPaths(50, 50, 50, 25, 25)
a3 = SolutionTable().findPaths(50, 50, 50, 25, 25)
assert a1 == a2 == a3
print("50x50, 50 bước ->", a1, "| ba cách DP cùng đáp án")
print("DP tiến chạy dưới 5 giây:", (t1 - t0) < 5)

50x50, 50 bước -> 276775132 | ba cách DP cùng đáp án
DP tiến chạy dưới 5 giây: True


## 12. Bộ test

Gồm: ví dụ đề, edge case, đối chiếu các cách bằng `assert` (kèm oracle liệt kê), và test ngẫu nhiên có seed trong ràng buộc.

In [25]:
sol = Solution()

# 1) Ví dụ đề
assert sol.findPaths(2, 2, 2, 0, 0) == 6
assert sol.findPaths(1, 3, 3, 0, 1) == 12
print("ví dụ đề OK")

# 2) Edge case
assert sol.findPaths(1, 1, 0, 0, 0) == 0          # maxMove = 0: không đi được bước nào
assert sol.findPaths(5, 5, 0, 2, 2) == 0
assert sol.findPaths(1, 1, 1, 0, 0) == 4          # lưới 1x1: bước nào cũng ra ngoài
assert sol.findPaths(1, 1, 3, 0, 0) == 4          # ra rồi thì không đi tiếp: vẫn 4
assert sol.findPaths(3, 3, 1, 1, 1) == 0          # ở giữa, 1 bước chưa ra được
assert sol.findPaths(3, 3, 1, 0, 0) == 2          # góc: 2 hướng ra
assert sol.findPaths(3, 3, 1, 0, 1) == 1          # cạnh: 1 hướng ra
assert sol.findPaths(1, 2, 1, 0, 0) == 3          # 1x2, ô trái: lên, xuống, trái
print("edge case OK")

# 3) Ví dụ lớn đã biết trước (kết quả khớp nhiều cách giải độc lập bên dưới)
assert sol.findPaths(8, 7, 16, 1, 5) == 102984580
print("ví dụ lớn: findPaths(8, 7, 16, 1, 5) =", sol.findPaths(8, 7, 16, 1, 5))

ví dụ đề OK
edge case OK
ví dụ lớn: findPaths(8, 7, 16, 1, 5) = 102984580


In [26]:
# 4) Đối chiếu các cách + oracle liệt kê trên nhiều lưới nhỏ (k <= 6 cho oracle)
random.seed(576)
solutions = [SolutionMemo(), SolutionTable(), SolutionForward(), SolutionMatrix(), Solution()]
fns = [find_paths_brute, find_paths_memo, find_paths_table, find_paths_forward, find_paths_matrix]
for _ in range(300):
    m, n = random.randint(1, 4), random.randint(1, 4)
    k = random.randint(0, 6)
    r, c = random.randrange(m), random.randrange(n)
    expected = count_by_enumeration(m, n, k, r, c)
    for f in fns:
        assert f(m, n, k, r, c) == expected, (f.__name__, m, n, k, r, c)
    for s in solutions:
        assert s.findPaths(m, n, k, r, c) == expected, (type(s).__name__, m, n, k, r, c)
print("300 input nhỏ: mọi cách khớp oracle liệt kê")

300 input nhỏ: mọi cách khớp oracle liệt kê


In [27]:
# 5) Ngẫu nhiên lớn có seed, đúng ràng buộc (1<=m,n<=50, 0<=maxMove<=50): so memo / bảng / tiến (bỏ brute và oracle vì quá chậm)
random.seed(2026)
for trial in range(12):
    m, n = random.randint(1, 50), random.randint(1, 50)
    k = random.randint(0, 50)
    r, c = random.randrange(m), random.randrange(n)
    a = SolutionMemo().findPaths(m, n, k, r, c)
    b = SolutionTable().findPaths(m, n, k, r, c)
    d = sol.findPaths(m, n, k, r, c)
    assert a == b == d, (m, n, k, r, c)
    assert 0 <= d < MOD
print("12 input lớn: ba cách DP khớp, kết quả trong [0, MOD)")

# 6) Matrix chỉ kiểm tra trên lưới nhỏ nhưng k lớn tới 50
for _ in range(20):
    m, n = random.randint(1, 3), random.randint(1, 3)
    k = random.randint(0, 50)
    r, c = random.randrange(m), random.randrange(n)
    assert SolutionMatrix().findPaths(m, n, k, r, c) == sol.findPaths(m, n, k, r, c)
print("nhân ma trận khớp DP trên lưới <= 3x3, k <= 50")

# 7) Test tính chất: tăng k thì đáp án (chưa mod) không giảm; đối xứng gương
def exact(m, n, k, r, c):                        # không mod: để thấy tính đơn điệu
    dp = [[0] * n for _ in range(m)]
    dp[r][c] = 1
    ans = 0
    for _ in range(k):
        nxt = [[0] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                    x, y = i + dr, j + dc
                    if 0 <= x < m and 0 <= y < n:
                        nxt[x][y] += dp[i][j]
                    else:
                        ans += dp[i][j]
        dp = nxt
    return ans

for m, n in [(3, 4), (5, 2)]:
    for r in range(m):
        for c in range(n):
            prev = 0
            for k in range(0, 9):
                cur = exact(m, n, k, r, c)
                assert cur >= prev
                prev = cur
                # đối xứng gương: lật trái-phải và trên-dưới cho cùng đáp án
                assert exact(m, n, k, r, n - 1 - c) == cur
                assert exact(m, n, k, m - 1 - r, c) == cur
print("đơn điệu theo k và đối xứng gương OK")

12 input lớn: ba cách DP khớp, kết quả trong [0, MOD)
nhân ma trận khớp DP trên lưới <= 3x3, k <= 50
đơn điệu theo k và đối xứng gương OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Kiểm tra `k == 0` **trước** khi kiểm tra ra ngoài

Bước đi cuối cùng đưa bóng ra ngoài vẫn là một đường hợp lệ. Nhưng nếu kiểm tra hết bước trước, lời gọi `dfs(ngoài, ngoài, 0)` bị trả `0` và ta mất đúng những đường dùng hết bước ở bước cuối.

In [28]:
def wrong_order(m, n, maxMove, sr, sc):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    @lru_cache(None)
    def dfs(i, j, k):
        if k == 0:                                  # SAI: kiểm tra k trước
            return 0
        if not (0 <= i < m and 0 <= j < n):
            return 1
        return sum(dfs(i + dr, j + dc, k - 1) for dr, dc in DIRS) % MOD
    return dfs(sr, sc, maxMove)

print("sai :", wrong_order(2, 2, 2, 0, 0), "(kỳ vọng 6)")
print("đúng:", Solution().findPaths(2, 2, 2, 0, 0))

sai : 2 (kỳ vọng 6)
đúng: 6


**Vì sao sai:** ở bước cuối, ta gọi `dfs(x, y, 0)` với `(x, y)` ngoài lưới; đáng ra phải trả `1` vì bóng **đã ra**. Kiểm tra `k == 0` trước nuốt mất những đường này. **Quy tắc:** *ra ngoài* được ưu tiên hơn *hết bước* (đã ra rồi thì số bước còn lại không còn quan trọng).

### Lỗi 2 — Bộ nhớ đệm thiếu `k`

Số đường ra phụ thuộc `k`. Nếu khoá cache chỉ là `(i, j)`, hai lời gọi cùng ô nhưng khác `k` bị nhầm.

In [29]:
def wrong_memo_missing_k(m, n, maxMove, sr, sc):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    memo = {}
    def dfs(i, j, k):
        if not (0 <= i < m and 0 <= j < n):
            return 1
        if k == 0:
            return 0
        if (i, j) in memo:                          # SAI: khoá thiếu k
            return memo[(i, j)]
        total = sum(dfs(i + dr, j + dc, k - 1) for dr, dc in DIRS) % MOD
        memo[(i, j)] = total
        return total
    return dfs(sr, sc, maxMove)

print("sai :", wrong_memo_missing_k(3, 3, 4, 1, 1), "(kỳ vọng", Solution().findPaths(3, 3, 4, 1, 1), ")")
print("đúng:", Solution().findPaths(3, 3, 4, 1, 1))

sai : 16 (kỳ vọng 52 )
đúng: 52


**Vì sao sai:** `dfs(i, j, 1)` (còn 1 bước) và `dfs(i, j, 3)` (còn 3 bước) là hai giá trị khác nhau, nhưng cache lưu chung một ô. **Quy tắc:** khoá memo phải chứa **toàn bộ tham số quyết định kết quả**.

### Lỗi 3 — Quên lấy dư (hoặc lấy dư chỉ một nơi)

Trong Python không tràn số nên đáp án vẫn "đúng toán học", nhưng **không đúng yêu cầu** (đề đòi giá trị modulo), và số khổng lồ làm chậm. Trong Java/C++ không lấy dư sẽ tràn số.

In [30]:
def wrong_no_mod(m, n, maxMove, sr, sc):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    dp = [[0] * n for _ in range(m)]
    dp[sr][sc] = 1
    ans = 0
    for _ in range(maxMove):
        nxt = [[0] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                for dr, dc in DIRS:
                    x, y = i + dr, j + dc
                    if 0 <= x < m and 0 <= y < n:
                        nxt[x][y] += dp[i][j]
                    else:
                        ans += dp[i][j]            # SAI: không % MOD
        dp = nxt
    return ans

wrong = wrong_no_mod(50, 50, 50, 25, 25)
right = Solution().findPaths(50, 50, 50, 25, 25)
print("sai :", wrong, "(số có", len(str(wrong)), "chữ số)")
print("đúng:", right)
print("khác nhau nhưng cùng dư:", wrong != right, wrong % MOD == right)

sai : 402483790271654788955008 (số có 24 chữ số)
đúng: 276775132
khác nhau nhưng cùng dư: True True


**Vì sao sai:** đề yêu cầu trả về kết quả **đã lấy dư**; số chưa lấy dư là một số có hàng chục chữ số, sẽ bị judge từ chối (và gây tràn ở ngôn ngữ khác). **Quy tắc:** cộng xong là `% MOD`; hoặc lấy dư một lần ở cuối *chỉ khi* ngôn ngữ có số lớn tuỳ ý và dữ liệu nhỏ.

### Lỗi 4 — Khởi tạo mảng 2D bằng `[[0] * n] * m`

Các hàng dùng chung một danh sách: sửa `nxt[x][y]` làm đổi cả cột `y` ở mọi hàng.

In [31]:
def wrong_alias(m, n, maxMove, sr, sc):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    dp = [[0] * n] * m                  # SAI: các hàng là cùng một đối tượng
    dp[sr][sc] = 1                      # đặt 1 ở cột sc của MỌI hàng!
    ans = 0
    for _ in range(maxMove):
        nxt = [[0] * n] * m             # SAI
        for i in range(m):
            for j in range(n):
                ways = dp[i][j]
                if ways == 0:
                    continue
                for dr, dc in DIRS:
                    x, y = i + dr, j + dc
                    if 0 <= x < m and 0 <= y < n:
                        nxt[x][y] += ways
                    else:
                        ans += ways
        dp = nxt
    return ans % MOD

print("sai :", wrong_alias(2, 2, 2, 0, 0), "(kỳ vọng 6)")
print("đúng:", Solution().findPaths(2, 2, 2, 0, 0))

sai : 20 (kỳ vọng 6)
đúng: 6


**Vì sao sai:** `[[0] * n] * m` lặp lại cùng một danh sách; `dp[sr][sc] = 1` biến cột `sc` của **mọi** hàng thành `1`, tức bóng "xuất hiện" ở nhiều chỗ. **Cách đúng:** `[[0] * n for _ in range(m)]`.

### Lỗi 5 — Hiểu "tối đa `maxMove`" thành "đúng `maxMove`"

Nếu chỉ lấy số đường ra ở **bước cuối** (hoặc chỉ tính `W` ở bước cuối), bạn bỏ sót mọi đường ra sớm.

In [32]:
def wrong_exactly(m, n, maxMove, sr, sc):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    dp = [[0] * n for _ in range(m)]
    dp[sr][sc] = 1
    last_exits = 0
    for _ in range(maxMove):
        nxt = [[0] * n for _ in range(m)]
        last_exits = 0                              # SAI: chỉ giữ số đường ra ở bước cuối
        for i in range(m):
            for j in range(n):
                for dr, dc in DIRS:
                    x, y = i + dr, j + dc
                    if 0 <= x < m and 0 <= y < n:
                        nxt[x][y] += dp[i][j]
                    else:
                        last_exits += dp[i][j]
        dp = nxt
    return last_exits % MOD

print("sai :", wrong_exactly(2, 2, 2, 0, 0), "(kỳ vọng 6; chỉ đếm đường ra ở bước 2)")
print("đúng:", Solution().findPaths(2, 2, 2, 0, 0))

sai : 4 (kỳ vọng 6; chỉ đếm đường ra ở bước 2)
đúng: 6


**Vì sao sai:** đường ra ở bước 1 (2 đường) bị bỏ, chỉ còn 4. **Quy tắc:** cộng **mọi** lần ra ngoài qua tất cả các bước (`ans += ...` trong cả vòng lặp, không gán lại).

## 14. Mẹo & nhận dạng pattern

- **"Đếm số đường đi / số cách" + lưới + giới hạn số bước nhỏ** → quy hoạch động theo `(ô, số bước)`. Dấu hiệu: tổng đường đi `4^k` nhưng số trạng thái `m·n·k` nhỏ.
- **Hai chiều nhìn DP:** *lùi* ("từ trạng thái này còn bao nhiêu cách tới đích", chính là đệ quy có nhớ) và *tiến* ("có bao nhiêu cách tới trạng thái này"). DP tiến thường dễ cuốn chiếu bộ nhớ.
- **Trạng thái hấp thụ:** "ra ngoài" là trạng thái kết thúc, đường đi không đi tiếp. Mô hình hoá bằng ô ảo hoặc bằng biến cộng dồn `ans` (như code của ta).
- **Thứ tự kiểm tra base case:** *thành công* trước, *hết tài nguyên* sau.
- **`mod`:** thấy "trả về modulo `10^9 + 7`" thì lấy dư ngay sau mỗi phép cộng/nhân.
- **Mỗi bước là phép biến đổi tuyến tính** → nếu `k` quá lớn, nghĩ tới nhân ma trận.
- Bài họ hàng: đếm đường đi trong lưới (62, 63, 64), Knight Probability in Chessboard (688), Number of Ways to Stay in the Same Place (1269).

## 15. Biến thể

### 15.1. Đúng `k` bước (không phải tối đa) — bóng phải ra ngoài ở đúng bước cuối

Chỉ giữ số đường ra **ở bước `k`**: dùng DP tiến và trả về `exits` của bước cuối. Có một cách nhìn hay: số đường ra ở từng bước `t` là một **dãy số** `E_1, E_2, ..., E_k`; đáp án gốc là tổng dãy, biến thể này là phần tử cuối. Hàm dưới trả về cả dãy.

In [33]:
def exits_per_step(m, n, maxMove, startRow, startColumn):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    dp = [[0] * n for _ in range(m)]
    dp[startRow][startColumn] = 1
    result = []
    for _ in range(maxMove):
        nxt = [[0] * n for _ in range(m)]
        exits = 0
        for i in range(m):
            for j in range(n):
                for dr, dc in DIRS:
                    x, y = i + dr, j + dc
                    if 0 <= x < m and 0 <= y < n:
                        nxt[x][y] += dp[i][j]
                    else:
                        exits += dp[i][j]
        result.append(exits)
        dp = nxt
    return result

seq = exits_per_step(1, 3, 3, 0, 1)
print(seq, "tổng =", sum(seq))                # [2, 6, 4] tổng 12
assert sum(seq) == Solution().findPaths(1, 3, 3, 0, 1)
assert sum(exits_per_step(5, 4, 12, 2, 1)) % MOD == Solution().findPaths(5, 4, 12, 2, 1)
print("đúng k bước:", seq[-1])

[2, 6, 4] tổng = 12
đúng k bước: 4


### 15.2. Số cách ở lại trong lưới sau đúng `k` bước

Đảo ngược câu hỏi: đếm các dãy hướng độ dài **đúng `k`** mà bóng **không bao giờ ra ngoài**. Đó chính là tổng các `dp[i][j]` sau `k` bước của DP tiến (bỏ phần `ans`). Ta dùng nó làm Bài 5 ở mục 17.

### 15.3. Số đường ra qua từng cạnh (trên/dưới/trái/phải)

Trong DP tiến, khi ô kề ngoài lưới, phân loại theo hướng `(dr, dc)` đang xét thay vì cộng chung vào `ans`: dùng bốn biến đếm (hoặc `dict`). Tổng bốn biến chính là đáp án gốc.

### 15.4. `maxMove` cực lớn (`10^18`) trên lưới nhỏ

Dùng nhân ma trận (Cách 5). Kiểm chứng: với `maxMove` vừa phải, nhân ma trận và DP cho cùng đáp án (đã test ở mục 12). Với `maxMove = 10^18` ta chỉ chạy được cách ma trận.

In [34]:
# Ma trận cho maxMove cực lớn trên lưới nhỏ; DP thường không chạy nổi
print(find_paths_matrix(1, 1, 10**18, 0, 0))         # lưới 1x1: bước đầu ra ngoài 4 cách, rồi hết
print(find_paths_matrix(2, 3, 10**18, 0, 0) < MOD)
# Kiểm chứng tính nhất quán: với k nhỏ khớp DP
assert find_paths_matrix(2, 3, 30, 1, 2) == Solution().findPaths(2, 3, 30, 1, 2)
print("ma trận khớp DP ở k = 30")

4
True
ma trận khớp DP ở k = 30


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy thuần | Chia theo bước đi đầu, 4 nhánh | `O(4^k)` | `O(k)` | Chỉ làm oracle cho `k ≤ 8` |
| 2. Đệ quy có nhớ | Cache theo `(i, j, k)` | `O(mnk)` | `O(mnk)` | Dễ viết nhất từ đệ quy; ổn với đề này |
| 3. Bảng DP lùi | `f[k][i][j]` từ `f[k-1]` | `O(mnk)` | `O(mnk)` | Cần đáp án cho **mọi** ô xuất phát; tránh đệ quy |
| 4. DP tiến cuốn chiếu | Đếm cách đang đứng ở mỗi ô theo từng bước | `O(mnk)` | `O(mn)` | **Lời giải chuẩn nộp LeetCode**; bộ nhớ nhỏ nhất |
| 5. Nhân ma trận | `A^k` trên vector (ô + trạng thái hấp thụ) | `O((mn)^3 log k)` | `O((mn)^2)` | `k` cực lớn, lưới rất nhỏ |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy kiểm tra:

```python
print(Solution().findPaths(1, 1, 1, 0, 0))
print(Solution().findPaths(1, 1, 0, 0, 0))
print((10**9 + 8) % (10**9 + 7))
```

In [35]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Lưới `1 x 2` (một hàng, hai cột), `maxMove = 2`, bóng ở `(0, 0)`. Hãy lập bảng "đang đứng ở đâu, bao nhiêu cách" qua từng bước và tính tổng số đường ra. Điền đáp án vào biến.

In [36]:
# Bài 2: điền kết quả tính tay
hand_exits_step1 = None    # số đường ra ở bước 1
hand_exits_step2 = None    # số đường ra ở bước 2
hand_answer = None         # tổng

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành DP tiến cuốn chiếu.

In [37]:
def exercise_3_find_paths(m: int, n: int, maxMove: int, startRow: int, startColumn: int) -> int:
    MOD_ = 10**9 + 7
    dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    # TODO 1: dp = bảng m x n toàn 0 (nhớ dùng list comprehension), dp[startRow][startColumn] = 1
    # TODO 2: ans = 0; lặp maxMove lần
    #     TODO 3: nxt = bảng m x n toàn 0 mới
    #     TODO 4: với mỗi ô có ways > 0 và mỗi hướng: ô kề trong lưới thì nxt[x][y] += ways, ngoài lưới thì ans += ways (nhớ % MOD_)
    #     TODO 5: dp = nxt
    # TODO 6: return ans
    return None


# assert exercise_3_find_paths(2, 2, 2, 0, 0) == 6
# assert exercise_3_find_paths(1, 3, 3, 0, 1) == 12
# assert exercise_3_find_paths(1, 1, 0, 0, 0) == 0

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, hãy tự viết `exercise_4_memo(m, n, maxMove, startRow, startColumn)` bằng đệ quy có nhớ. Chú ý: thứ tự base case và khoá cache.

In [38]:
def exercise_4_memo(m: int, n: int, maxMove: int, startRow: int, startColumn: int) -> int:
    # TODO
    pass


# assert exercise_4_memo(2, 2, 2, 0, 0) == 6
# assert exercise_4_memo(1, 3, 3, 0, 1) == 12
# assert exercise_4_memo(8, 7, 16, 1, 5) == 102984580

### Bài 5 — Mini challenge: số cách ở lại trong lưới (khó)

Viết `exercise_5_stay_inside(m, n, k, startRow, startColumn)` trả về (modulo `10^9 + 7`) số dãy hướng độ dài **đúng `k`** mà bóng **không bao giờ ra khỏi lưới**.

```python
exercise_5_stay_inside(2, 2, 1, 0, 0)  -> 2     # chỉ đi xuống hoặc sang phải
exercise_5_stay_inside(1, 1, 1, 0, 0)  -> 0     # mọi hướng đều ra ngoài
exercise_5_stay_inside(3, 3, 0, 1, 1)  -> 1     # không đi bước nào: đúng 1 dãy rỗng
```

In [39]:
def exercise_5_stay_inside(m: int, n: int, k: int, startRow: int, startColumn: int) -> int:
    # TODO
    pass


# assert exercise_5_stay_inside(2, 2, 1, 0, 0) == 2
# assert exercise_5_stay_inside(1, 1, 1, 0, 0) == 0
# assert exercise_5_stay_inside(3, 3, 0, 1, 1) == 1

## 18. Gợi ý 3 mức

**Bài 3 / Bài 4**

- *Mức 1 (nhẹ):* Từ một ô, bóng đi 4 hướng. Mỗi hướng hoặc rơi vào ô khác trong lưới, hoặc ra ngoài. Mỗi trường hợp xử lý thế nào?
- *Mức 2 (vừa):* DP tiến: `dp[i][j]` là "bao nhiêu cách đang đứng ở ô này". Đi ra ngoài thì cộng vào `ans`, đi vào ô kề thì cộng vào `nxt`. Đệ quy: `dfs(i, j, k)` — ra ngoài trả `1` **trước**, rồi mới xét `k == 0` trả `0`.
- *Mức 3 (gần đáp án):* DP tiến: `nxt[x][y] = (nxt[x][y] + ways) % mod`, `ans = (ans + ways) % mod`, rồi `dp = nxt`. Đệ quy: khoá cache là `(i, j, k)`.

**Bài 5**

- *Mức 1:* Bóng bị cấm ra ngoài. Phần "cộng vào `ans`" trong DP tiến còn cần không?
- *Mức 2:* Dùng DP tiến nhưng **bỏ qua** các hướng ra ngoài. Sau `k` bước, `dp` cho biết có bao nhiêu dãy kết thúc ở mỗi ô.
- *Mức 3:* Đáp án = tổng mọi phần tử của `dp` sau đúng `k` bước (nhớ `% mod`). Trường hợp `k = 0`: `dp` ban đầu có tổng `1`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

- Dòng 1: lưới `1x1`, 1 bước: cả 4 hướng đều ra ngoài → `4`.
- Dòng 2: `maxMove = 0` → `0`.
- Dòng 3: `(10^9 + 8) % (10^9 + 7) = 1`.

In [40]:
assert Solution().findPaths(1, 1, 1, 0, 0) == 4
assert Solution().findPaths(1, 1, 0, 0, 0) == 0
assert (10**9 + 8) % (10**9 + 7) == 1
print("Bài 1 OK")

Bài 1 OK


### Đáp án Bài 2

Lưới `1 x 2`, bóng ở `(0,0)`.

- Bước 1: từ `(0,0)`: lên, xuống, trái → ra ngoài (3 đường); phải → `(0,1)`. `hand_exits_step1 = 3`.
- Bước 2: có 1 cách đang ở `(0,1)`: lên, xuống, phải → ra ngoài (3 đường); trái → `(0,0)` (còn trong lưới nhưng hết bước). `hand_exits_step2 = 3`.
- Tổng `6`.

In [41]:
hand_exits_step1, hand_exits_step2, hand_answer = 3, 3, 6
assert exits_per_step(1, 2, 2, 0, 0) == [hand_exits_step1, hand_exits_step2]
assert hand_answer == Solution().findPaths(1, 2, 2, 0, 0) == count_by_enumeration(1, 2, 2, 0, 0)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [42]:
def exercise_3_solution(m: int, n: int, maxMove: int, startRow: int, startColumn: int) -> int:
    MOD_ = 10**9 + 7
    dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    dp = [[0] * n for _ in range(m)]
    dp[startRow][startColumn] = 1
    ans = 0
    for _ in range(maxMove):
        nxt = [[0] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                ways = dp[i][j]
                if ways == 0:
                    continue
                for dr, dc in dirs:
                    x, y = i + dr, j + dc
                    if 0 <= x < m and 0 <= y < n:
                        nxt[x][y] = (nxt[x][y] + ways) % MOD_
                    else:
                        ans = (ans + ways) % MOD_
        dp = nxt
    return ans

assert exercise_3_solution(2, 2, 2, 0, 0) == 6
assert exercise_3_solution(1, 3, 3, 0, 1) == 12
assert exercise_3_solution(1, 1, 0, 0, 0) == 0
random.seed(3)
for _ in range(100):
    m, n, k = random.randint(1, 6), random.randint(1, 6), random.randint(0, 12)
    r, c = random.randrange(m), random.randrange(n)
    assert exercise_3_solution(m, n, k, r, c) == Solution().findPaths(m, n, k, r, c)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [43]:
def exercise_4_solution(m: int, n: int, maxMove: int, startRow: int, startColumn: int) -> int:
    MOD_ = 10**9 + 7
    dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    @lru_cache(None)
    def dfs(i, j, k):
        if not (0 <= i < m and 0 <= j < n):     # ra ngoài trước
            return 1
        if k == 0:                              # hết bước sau
            return 0
        total = 0
        for dr, dc in dirs:
            total = (total + dfs(i + dr, j + dc, k - 1)) % MOD_
        return total

    return dfs(startRow, startColumn, maxMove)

assert exercise_4_solution(2, 2, 2, 0, 0) == 6
assert exercise_4_solution(1, 3, 3, 0, 1) == 12
assert exercise_4_solution(8, 7, 16, 1, 5) == 102984580
random.seed(4)
for _ in range(100):
    m, n, k = random.randint(1, 8), random.randint(1, 8), random.randint(0, 20)
    r, c = random.randrange(m), random.randrange(n)
    assert exercise_4_solution(m, n, k, r, c) == Solution().findPaths(m, n, k, r, c)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [44]:
def exercise_5_solution(m: int, n: int, k: int, startRow: int, startColumn: int) -> int:
    MOD_ = 10**9 + 7
    dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    dp = [[0] * n for _ in range(m)]
    dp[startRow][startColumn] = 1
    for _ in range(k):
        nxt = [[0] * n for _ in range(m)]
        for i in range(m):
            for j in range(n):
                ways = dp[i][j]
                if ways == 0:
                    continue
                for dr, dc in dirs:
                    x, y = i + dr, j + dc
                    if 0 <= x < m and 0 <= y < n:
                        nxt[x][y] = (nxt[x][y] + ways) % MOD_
        dp = nxt
    return sum(sum(row) for row in dp) % MOD_

assert exercise_5_solution(2, 2, 1, 0, 0) == 2
assert exercise_5_solution(1, 1, 1, 0, 0) == 0
assert exercise_5_solution(3, 3, 0, 1, 1) == 1

# Kiểm chứng bằng liệt kê: số dãy độ dài đúng k không bao giờ ra ngoài
def stay_by_enumeration(m, n, k, sr, sc):
    DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    cnt = 0
    for seq in product(range(4), repeat=k):
        i, j = sr, sc
        ok = True
        for d in seq:
            i, j = i + DIRS[d][0], j + DIRS[d][1]
            if not (0 <= i < m and 0 <= j < n):
                ok = False
                break
        cnt += ok
    return cnt

for m in range(1, 4):
    for n in range(1, 4):
        for k in range(0, 6):
            for r in range(m):
                for c in range(n):
                    assert exercise_5_solution(m, n, k, r, c) == stay_by_enumeration(m, n, k, r, c)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def findPaths(self, m: int, n: int, maxMove: int, startRow: int, startColumn: int) -> int:
        mod = 10**9 + 7
        dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        dp = [[0] * n for _ in range(m)]          # dp[i][j]: số cách đang đứng ở (i, j)
        dp[startRow][startColumn] = 1
        ans = 0
        for _ in range(maxMove):
            nxt = [[0] * n for _ in range(m)]
            for i in range(m):
                for j in range(n):
                    ways = dp[i][j]
                    if ways == 0:
                        continue
                    for dr, dc in dirs:
                        x, y = i + dr, j + dc
                        if 0 <= x < m and 0 <= y < n:
                            nxt[x][y] = (nxt[x][y] + ways) % mod   # còn trong lưới
                        else:
                            ans = (ans + ways) % mod               # ra ngoài: một đường hợp lệ
            dp = nxt
        return ans
```

Câu thần chú: **"Ra ngoài thì cộng vào đáp án, ở trong thì chuyển sang bảng mới; xong mỗi bước đổi bảng."**

### Checklist trước khi submit

- [ ] Hiểu "tối đa `maxMove`": cộng số đường ra ở **mọi** bước, không chỉ bước cuối.
- [ ] Ra ngoài biên được xét **trước** hết bước (nếu dùng đệ quy).
- [ ] Bộ nhớ đệm có đủ `(i, j, k)`; mảng 2D tạo bằng list comprehension.
- [ ] `% mod` sau mỗi phép cộng; kết quả cuối nằm trong `[0, mod)`.
- [ ] Đã thử `maxMove = 0`, lưới `1x1`, ô góc, ô cạnh, ô giữa.
- [ ] Đã đối chiếu oracle liệt kê trên input nhỏ.
- [ ] Skeleton đúng: `class Solution`, method `findPaths(self, m, n, maxMove, startRow, startColumn)`.